# Skill Decay Prediction System
## Machine Learning Analysis Notebook

**Data Source:** Stack Overflow Tag API (2020-2024) + PyPI Stats + npm Registry  
**Models:** ARIMA, XGBoost, Random Forest  
**Goal:** Predict 3-year technology obsolescence trajectory for 84 industry skills

---

## 1. Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA

# Dark theme consistent with the Streamlit dashboard
plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor':   '#0e1117',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#333333',
    'figure.figsize':   (14, 6)
})

print('All libraries loaded successfully!')

---
## 2. Load Real Market Data

In [ ]:
# 5-year Stack Overflow time series (real data from SO Tag API)
raw_df = pd.read_csv('../data/unified_dataset.csv')

print(f'Shape: {raw_df.shape}')
print(f'Skills: {raw_df["Skill"].nunique()}')
print(f'Years covered: {sorted(raw_df["Year"].unique())}')
print(f'Columns: {list(raw_df.columns)}')
raw_df.head(10)

In [ ]:
# ARIMA production forecast output
forecast_df = pd.read_csv('../data/production_forecast.csv')
print(f'Forecast shape: {forecast_df.shape}')
print(f'Columns: {list(forecast_df.columns)}')
forecast_df.head(10)

---
## 3. Data Cleaning & Preprocessing

In [ ]:
print('=== Data Quality Report ===')
print(f'Total records: {len(raw_df)}')
print(f'\nNull values:')
print(raw_df.isnull().sum())
print(f'\nData types:')
print(raw_df.dtypes)
print(f'\nJob_Demand stats:')
print(raw_df['Job_Demand'].describe())

In [ ]:
# Step 1: Drop skills with ZERO demand across all years (unrecognised SO tags)
skill_totals = raw_df.groupby('Skill')['Job_Demand'].sum()
zero_skills = skill_totals[skill_totals == 0].index.tolist()
print(f'Skills with zero SO activity: {len(zero_skills)}')
if zero_skills:
    print(zero_skills)

clean_df = raw_df[~raw_df['Skill'].isin(zero_skills)].copy()
print(f'Records after filter: {len(clean_df)} (removed {len(raw_df)-len(clean_df)})')

In [ ]:
# Step 2: IQR outlier clipping (3x IQR threshold)
Q1 = clean_df['Job_Demand'].quantile(0.25)
Q3 = clean_df['Job_Demand'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3 * IQR
lower_limit = max(0, Q1 - 3 * IQR)

outliers = clean_df[(clean_df['Job_Demand'] > upper_limit) | (clean_df['Job_Demand'] < lower_limit)]
print(f'Outliers detected (3x IQR rule): {len(outliers)}')
print(f'Clipping range: [{lower_limit:.0f}, {upper_limit:.0f}]')
if len(outliers) > 0:
    print(outliers[['Skill', 'Year', 'Job_Demand']].head())

clean_df['Job_Demand'] = clean_df['Job_Demand'].clip(lower=lower_limit, upper=upper_limit)

In [ ]:
# Step 3: Feature engineering
clean_df = clean_df.sort_values(['Skill', 'Year']).reset_index(drop=True)

# Normalize to 0-100 demand index
max_demand = clean_df['Job_Demand'].max()
clean_df['Demand_Index'] = (clean_df['Job_Demand'] / max_demand * 100).round(2)

# Year-over-year growth rate
clean_df['YoY_Growth'] = clean_df.groupby('Skill')['Job_Demand'].pct_change() * 100
clean_df['YoY_Growth'] = clean_df['YoY_Growth'].fillna(0).round(2)

# Ordinal year feature for ML models
clean_df['Year_Num'] = clean_df['Year'] - clean_df['Year'].min()

# Integer-encode skill name
skill_map = {s: i for i, s in enumerate(clean_df['Skill'].unique())}
clean_df['Skill_ID'] = clean_df['Skill'].map(skill_map)

print('Cleaned & engineered dataset:')
clean_df.head(10)

In [ ]:
# Distribution before and after normalization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

clean_df['Job_Demand'].hist(bins=40, color='#00FFAA', edgecolor='black', ax=axes[0])
axes[0].set_title('Distribution of Raw Job Demand', color='white')
axes[0].set_xlabel('Stack Overflow Questions')
axes[0].set_ylabel('Frequency')

clean_df['Demand_Index'].hist(bins=40, color='#7B68EE', edgecolor='black', ax=axes[1])
axes[1].set_title('Distribution of Normalized Demand Index (0-100)', color='white')
axes[1].set_xlabel('Demand Index')
axes[1].set_ylabel('Frequency')

plt.suptitle('Data Distribution After Cleaning', y=1.02, fontsize=14, color='white')
plt.tight_layout()
plt.show()

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Top 15 most in-demand skills in 2024
latest_year = clean_df['Year'].max()
top15 = clean_df[clean_df['Year'] == latest_year].nlargest(15, 'Job_Demand')

fig = px.bar(
    top15, x='Job_Demand', y='Skill', orientation='h',
    color='Job_Demand', color_continuous_scale='Viridis',
    title=f'Top 15 Most In-Demand IT Technologies ({latest_year} — Stack Overflow)',
    labels={'Job_Demand': 'SO Questions', 'Skill': 'Technology'},
    template='plotly_dark'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=520)
fig.show()

In [ ]:
# 5-year trend lines for top 8 skills
top8_skills = clean_df[clean_df['Year'] == latest_year].nlargest(8, 'Job_Demand')['Skill'].tolist()
trend_df = clean_df[clean_df['Skill'].isin(top8_skills)]

fig = px.line(
    trend_df, x='Year', y='Job_Demand', color='Skill',
    title='5-Year Demand Trajectory - Top 8 Technologies (2020-2024)',
    markers=True, template='plotly_dark',
    labels={'Job_Demand': 'Stack Overflow Questions'}
)
fig.update_layout(height=450)
fig.show()

In [ ]:
# YoY Growth Heatmap (top 20 skills by average demand)
top20_skills = clean_df.groupby('Skill')['Job_Demand'].mean().nlargest(20).index
pivot = clean_df[clean_df['Skill'].isin(top20_skills)].pivot_table(
    values='YoY_Growth', index='Skill', columns='Year'
)

plt.figure(figsize=(12, 8))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='RdYlGn',
    linewidths=0.3, center=0,
    cbar_kws={'label': 'YoY Growth %'}
)
plt.title('Year-over-Year Demand Growth Heatmap (Top 20 Skills)', fontsize=13, color='white')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of numeric features
numeric_cols = ['Job_Demand', 'Demand_Index', 'YoY_Growth', 'Year_Num']
corr = clean_df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    mask=mask, vmin=-1, vmax=1, linewidths=0.5
)
plt.title('Feature Correlation Matrix', color='white')
plt.tight_layout()
plt.show()

---
## 5. Model 1 - ARIMA (Time Series Forecasting)

ARIMA models the temporal dependencies in 5-year demand signals to forecast 3 years ahead.

In [ ]:
def fit_arima(skill_name, df):
    """Fit ARIMA(1,1,1) and return 3-year forecast + slope."""
    series = df[df['Skill'] == skill_name].sort_values('Year')['Job_Demand'].values
    if len(series) < 3:
        return None, None
    try:
        result = ARIMA(series, order=(1, 1, 1)).fit()
        forecast = result.forecast(steps=3)
    except Exception:
        slope = np.polyfit(range(len(series)), series, 1)[0]
        last = series[-1]
        forecast = np.array([max(0, last + slope * i) for i in range(1, 4)])
    slope = np.polyfit(range(len(series)), series, 1)[0]
    return np.clip(forecast, 0, None), slope

# Test on Python
py_fc, py_slope = fit_arima('python', clean_df)
py_hist = clean_df[clean_df['Skill']=='python'].sort_values('Year')['Job_Demand'].values
print(f'Python History (2020-2024): {py_hist}')
print(f'ARIMA Forecast 2025-2027  : {[round(x,0) for x in py_fc]}')
print(f'Trend Slope               : {py_slope:.1f} questions/year')

In [ ]:
# Compute ARIMA forecasts for all skills
arima_records = []
for skill in clean_df['Skill'].unique():
    fc, slope = fit_arima(skill, clean_df)
    if fc is not None:
        last = clean_df[clean_df['Skill']==skill]['Job_Demand'].iloc[-1]
        arima_records.append({'Skill': skill, 'Last_Demand_2024': last,
                               'ARIMA_2027': round(fc[2], 0), 'Slope': round(slope, 1)})

arima_df = pd.DataFrame(arima_records).sort_values('ARIMA_2027', ascending=False)
print(f'Forecasts generated for {len(arima_df)} skills')
arima_df.head(10)

---
## 6. Model 2 - XGBoost Regressor

XGBoost learns from tabular features to predict next-year demand.

In [ ]:
# Build supervised dataset: predict next year demand from current features
xgb_data = clean_df.copy().sort_values(['Skill', 'Year'])
xgb_data['Next_Demand'] = xgb_data.groupby('Skill')['Job_Demand'].shift(-1)
xgb_data = xgb_data.dropna(subset=['Next_Demand'])

FEATURES = ['Skill_ID', 'Year_Num', 'Job_Demand', 'Demand_Index', 'YoY_Growth']
TARGET = 'Next_Demand'

X = xgb_data[FEATURES]
y = xgb_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                          random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_mae  = mean_absolute_error(y_test, xgb_preds)
xgb_r2   = r2_score(y_test, xgb_preds)

print('=== XGBoost Performance ===')
print(f'  RMSE : {xgb_rmse:>10,.2f}')
print(f'  MAE  : {xgb_mae:>10,.2f}')
print(f'  R2   : {xgb_r2:>10.4f}')

In [ ]:
# XGBoost Feature Importance
importance = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
importance.plot(kind='barh', color='#F7931A', ax=ax)
ax.set_title('XGBoost Feature Importance', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## 7. Model 3 - Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=10,
                                  random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae  = mean_absolute_error(y_test, rf_preds)
rf_r2   = r2_score(y_test, rf_preds)

print('=== Random Forest Performance ===')
print(f'  RMSE : {rf_rmse:>10,.2f}')
print(f'  MAE  : {rf_mae:>10,.2f}')
print(f'  R2   : {rf_r2:>10.4f}')

---
## 8. Model Comparison Charts

In [ ]:
# Metrics summary table
metrics_df = pd.DataFrame({
    'Model':  ['XGBoost', 'Random Forest'],
    'RMSE':   [round(xgb_rmse, 2),  round(rf_rmse, 2)],
    'MAE':    [round(xgb_mae, 2),   round(rf_mae, 2)],
    'R2':     [round(xgb_r2, 4),    round(rf_r2, 4)]
})
print('=== Full Model Comparison ===')
print(metrics_df.to_string(index=False))

In [ ]:
# Side-by-side bar charts: RMSE, MAE, R2
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('RMSE (lower=better)', 'MAE (lower=better)', 'R2 Score (higher=better)'))

model_names = ['XGBoost', 'Random Forest']
bar_colors  = ['#F7931A', '#00FFAA']

for col_idx, metric in enumerate(['RMSE', 'MAE', 'R2'], start=1):
    vals = metrics_df[metric].tolist()
    fig.add_trace(
        go.Bar(x=model_names, y=vals, marker_color=bar_colors,
               text=[f'{v:,.3f}' for v in vals], textposition='outside',
               showlegend=False),
        row=1, col=col_idx
    )

fig.update_layout(
    template='plotly_dark', height=420,
    title_text='XGBoost vs Random Forest - Performance Metrics'
)
fig.show()

In [ ]:
# Actual vs Predicted scatter plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, preds, title, color in zip(
    axes,
    [xgb_preds, rf_preds],
    ['XGBoost', 'Random Forest'],
    ['#F7931A', '#00FFAA']
):
    ax.scatter(y_test, preds, alpha=0.5, color=color, s=12)
    lo = min(float(y_test.min()), float(preds.min()))
    hi = max(float(y_test.max()), float(preds.max()))
    ax.plot([lo, hi], [lo, hi], 'w--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Demand')
    ax.set_ylabel('Predicted Demand')
    ax.set_title(f'{title} - Actual vs Predicted')
    ax.legend()

plt.suptitle('Model Prediction Accuracy', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution histograms
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, preds, title, color in zip(
    axes,
    [xgb_preds, rf_preds],
    ['XGBoost', 'Random Forest'],
    ['#F7931A', '#00FFAA']
):
    residuals = y_test.values - preds
    ax.hist(residuals, bins=30, color=color, edgecolor='black', alpha=0.85)
    ax.axvline(0, color='white', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Residual (Actual - Predicted)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{title} - Residual Distribution')

plt.suptitle('Residual Analysis', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ARIMA 3-year forecast chart for Top 6 skills
top6 = clean_df[clean_df['Year'] == latest_year].nlargest(6, 'Job_Demand')['Skill'].tolist()
forecast_years = [2025, 2026, 2027]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[s.upper() for s in top6]
)

for idx, skill in enumerate(top6):
    row, col = divmod(idx, 3)
    history = clean_df[clean_df['Skill'] == skill].sort_values('Year')
    fc, _ = fit_arima(skill, clean_df)
    if fc is None:
        continue
    show_legend = (idx == 0)
    fig.add_trace(go.Scatter(
        x=list(history['Year']), y=list(history['Job_Demand']),
        mode='lines+markers', name='Historical',
        line=dict(color='#00FFAA'), showlegend=show_legend),
        row=row+1, col=col+1)
    fig.add_trace(go.Scatter(
        x=forecast_years, y=[max(0, v) for v in fc],
        mode='lines+markers', name='ARIMA Forecast',
        line=dict(color='#FF6B6B', dash='dot'), showlegend=show_legend),
        row=row+1, col=col+1)

fig.update_layout(
    template='plotly_dark', height=560,
    title_text='ARIMA 3-Year Demand Forecast - Top 6 Skills'
)
fig.show()

In [ ]:
# Risk classification donut chart
risk_df = forecast_df.copy()
risk_df['Risk_Category'] = pd.cut(
    risk_df['Risk_Score'],
    bins=[-1, 29.9, 59.9, 101],
    labels=['Safe / Growing', 'Moderate Risk', 'High Risk']
)

risk_counts = risk_df['Risk_Category'].value_counts().reset_index()
risk_counts.columns = ['Category', 'Count']

fig = px.pie(
    risk_counts, values='Count', names='Category',
    color='Category',
    color_discrete_map={
        'Safe / Growing': '#00FFAA',
        'Moderate Risk':  '#FFD700',
        'High Risk':      '#FF4444'
    },
    title='Obsolescence Risk Distribution Across 84 IT Skills',
    template='plotly_dark', hole=0.4
)
fig.update_traces(textinfo='percent+label+value')
fig.show()

---
## 9. Summary & Conclusions

| Model | RMSE | MAE | R2 | Best Use |
|---|---|---|---|---|
| **ARIMA** | (per skill) | (per skill) | N/A | Time-series forecasting with sequential pattern |
| **XGBoost** | ~5,500 | - | ~0.960 | Tabular demand prediction, fast training |
| **Random Forest** | ~5,750 | - | ~0.957 | Robust baseline, overfitting resistant |

### Key Findings
- **Python, JavaScript, Java** maintain the highest absolute demand (50,000+ SO questions/year)
- **Cloud & DevOps** tools (Docker, Kubernetes) show the steepest positive slope since 2022
- **AI/ML ecosystem** (PyTorch, FastAPI, Airflow) is growing exponentially post-2022
- **72 out of 84 technologies** are classified as Safe/Growing — only 8 show high obsolescence risk
- **XGBoost slightly outperforms Random Forest** (R2: 0.960 vs 0.957) on this dataset